# Building a new `cafpybara` analysis, file by file

Covers every file in `analyses/_template/`, `__init__.py` first for the
overall shape, then each file it re-exports. For the order you'd actually
*write* these in from scratch, see README.md's "Reference: the override
contract" instead -- `__init__.py` goes last there, since its real content
depends on everything else existing first.

- Needs no real Fermilab/EAF data -- `_template/` is self-contained.
- To start a new analysis: copy `_template/` to `analyses/<yourname>/`,
  fill in every `# TODO`.

In [ ]:
%load_ext autoreload
%autoreload 2

import os, sys
# This notebook lives 4 directories below the one containing `cafpybara/`
# (examples/ -> _template/ -> analyses/ -> cafpybara/ -> parent) -- Jupyter
# sets cwd to the notebook's own directory, so make that parent importable
# without hardcoding anyone's personal path.
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "../../../..")))

import pandas as pd
import cafpybara.analyses._template as ana
from cafpybara import core

print("Imported the template analysis cleanly -- it's a real, importable")
print("(if physically inert) analysis, not just documentation.")

## 1. `__init__.py` -- the re-export chain

- `from ...core.X import *` for every `core` submodule, then `from .X
  import *` for this analysis's own `analysis`/`preprocess`/`io`/
  `plotting`/`funcs` -- the second block **shadows** the first with real,
  topology-specific defaults.
- Get this exact, unmodified, when copying the template.
- Real bugs: nueCC's and hnlpi0's `__init__.py` were both independently
  found missing one `core.*` re-export line (`core.io`/`funcs`/
  `plotting`/`selection`/`physics`) -- live `AttributeError`s the moment a
  notebook used the missing name.

Check it below -- every one of these should resolve:

In [ ]:
core_modules_to_check = ['io', 'funcs', 'plotting', 'selection', 'physics', 'syst', 'classes', 'preprocess', 'detvar']
for mod_name in core_modules_to_check:
    core_mod = getattr(core, mod_name)
    missing = [n for n in getattr(core_mod, '__all__', []) if not hasattr(ana, n)]
    status = 'OK' if not missing else f'MISSING: {missing}'
    print(f"core.{mod_name:12s} -> {status}")

## 2. `config.py` -- your file paths

- Constants only -- detvar store paths, in-time-cosmic file paths.
- Placeholders are obviously-broken strings, not `None` -- fails loud
  (`FileNotFoundError`) if used unfilled, instead of silently propagating.

In [ ]:
print("DETVAR_DICT_FILES:", ana.config.DETVAR_DICT_FILES)
print("INTIME_FILE:", ana.config.INTIME_FILE)

## 3. `analysis.py` -- categories and cuts

- `signal_categories`/`signal_dict`: category dict for stacked plots and
  the `signal` truth column.
- `DEFAULT_CUTS`: a list of `CutSpec` (threshold `variable=`/`min=`/`max=`,
  or a full `fn=` mask). `select()` ANDs them in order.
- Template values are toy placeholders -- illustrative shape, not real
  column names/thresholds.

In [ ]:
print("signal_categories keys:", list(ana.signal_categories.keys()))
print("signal_dict:", ana.signal_dict)
print("\nDEFAULT_CUTS:")
for c in ana.DEFAULT_CUTS:
    print(f"  {c.name!r}: {c.label}")

## 4. `preprocess.py` -- the single most consequential file here

- `core.preprocess.preprocess_mc`/`preprocess_data` are deliberately
  generic no-ops -- correct only if your topology truly needs nothing
  extra.
- Real bugs: nueCC's `load_mc`/`load_data`, then hnlpi0's `load_mchnl`,
  were each independently found wired to the core no-op instead of a real
  bundler -- silently wrong preprocessing on every load, caught only by
  noticing plotted numbers didn't match a known-good baseline.
- Pattern to extend: real base → your fixes → `add_variables`.

In [ ]:
import inspect
print(inspect.getsource(ana.preprocess.preprocess_mc_real))

## 5. `io.py` -- `load_mc`/`load_data` wrappers

- Thin wrappers filling in this analysis's real `rec_key`/`preprocess_fn`/
  `define_signal_fn`.
- `core.io.load_mc` has **no default** for any of the three -- deliberate,
  so a new analysis can't silently inherit another topology's `rec_key`
  (`'nuecc'` vs. `'rec'`).

In [ ]:
import inspect
sig = inspect.signature(core.io.load_mc)
print("core.io.load_mc's rec_key/preprocess_fn/define_signal_fn have no default:")
for name in ('rec_key', 'preprocess_fn', 'define_signal_fn'):
    p = sig.parameters[name]
    has_default = p.default is not inspect.Parameter.empty
    print(f"  {name}: {'default=' + repr(p.default) if has_default else 'no default (required)'}")
print("\nYour analysis's io.py fills these in -- REC_KEY placeholder:", ana.io.REC_KEY)

## 6. `plotting.py` -- category injection

- Same eager-resolution pattern as `funcs.py` below, applied to plot
  categories.
- `core.plot_var` has no category default -- raises if one can't be
  resolved. Your wrapper fills in the real default before calling `core`.

In [ ]:
try:
    core.plotting.plot_var(df=pd.DataFrame(), var='x', bins=[0, 1])
except Exception as e:
    print(f"core.plot_var with no categories given raises immediately: {type(e).__name__}: {e}")

## 7. `funcs.py` -- `systs_input()`, and why there's no `get_total_cov` here

- Real bug: `core/plotting.py`'s `systs=` handling called
  `core.funcs.get_total_cov` directly; notebook drill-down cells called a
  per-analysis `get_total_cov` with different resolved defaults. The two
  silently disagreed -- a plotted error band was missing an uncertainty
  source its own drill-down table had, with no visible sign anything was
  wrong.
- Rule: `core.funcs.get_total_cov` is the *only* `get_total_cov` -- no
  analysis defines its own. `systs_input(...)` fully resolves every
  default into a concrete `SystematicsInput` before anything reaches
  `core`, so every caller necessarily agrees.

In [ ]:
print("ana.get_total_cov IS core.funcs.get_total_cov:", ana.get_total_cov is core.funcs.get_total_cov)
print("(if this were ever False for a new analysis, that's the bug reintroduced)\n")

si = ana.systs_input(1e20, detvar_dict={'placeholder': True})
print("systs_input() resolved cuts to DEFAULT_CUTS:", si.cuts is ana.DEFAULT_CUTS)
print("resolved uncertainty_keys:", si.uncertainty_keys)
print("\nto_kwargs() ready to unpack into get_total_cov:")
print(list(si.to_kwargs().keys()))

## 8. Before you ship it: a manual conformance pass

`scripts/verify_new_analysis.py <name>` runs this automatically (plus two
checks this manual version can't: a no-op-preprocessing-default audit, and
an ast-based sweep of every example notebook's `systs_input`/
`SystematicsInput`/`PlottingConfig` calls for dead kwargs). If your analysis
has a loader that needs real preprocessing, add an entry for it to that
script's `NOOP_DEFAULT_CHECKS` table -- it's curated per-analysis, not
derived automatically. The version below is the same idea as the script's
first two checks, inline, so you can see how it works without leaving the
notebook:

In [ ]:
def conformance_check(analysis_module):
    """Run this against your real analysis module, not the template."""
    problems = []

    for mod_name in ['io', 'funcs', 'plotting', 'selection', 'physics', 'syst', 'classes', 'preprocess', 'detvar']:
        core_mod = getattr(core, mod_name)
        missing = [n for n in getattr(core_mod, '__all__', []) if not hasattr(analysis_module, n)]
        if missing:
            problems.append(f"core.{mod_name} not fully re-exported: missing {missing}")

    if analysis_module.get_total_cov is not core.funcs.get_total_cov:
        problems.append("get_total_cov has been overridden -- delete it, keep only systs_input()")

    if not hasattr(analysis_module, 'systs_input'):
        problems.append("no systs_input() factory found")

    if problems:
        print(f"{len(problems)} problem(s) found:")
        for p in problems:
            print(f"  - {p}")
    else:
        print("No problems found by this (non-exhaustive) check.")
    return problems


conformance_check(ana)

## Next steps

1. Copy `analyses/_template/` to `analyses/<yourname>/`.
2. Fill in every `# TODO` in `config.py`, `analysis.py`, `preprocess.py`,
   `io.py`, `plotting.py`, `funcs.py` (README's write order -- `__init__.py`
   last, once everything it re-exports actually exists).
3. If you'll build detvar stores: add an entry for your analysis in
   `core/detvar/process_detvars.py`'s `_SLC_KEY`/`_default_preprocess_fn`/
   `_selection_fn_map` -- the one place in `core/` that needs an edit.
4. Run `python scripts/verify_new_analysis.py <yourname>` (section 8) against
   your real module.
5. See `README.md`'s "Reference: the override contract" and "Common
   mistakes" sections for the fuller checklist.